In [1]:
import pandas as pd
import numpy as np
import joblib

In [2]:
df = pd.read_csv(
    "/content/nassau_candy_cleaned.csv"
)

print(df.shape)

(10194, 32)


In [3]:
product_factory = pd.read_csv(
    "/content/product_factory_mapping.csv"
)

print(product_factory)

                         Product Name            Factory
0   Wonka Bar - Nutty Crunch Surprise      Lot's O' Nuts
1           Wonka Bar - Fudge Mallows      Lot's O' Nuts
2      Wonka Bar -Scrumdiddlyumptious      Lot's O' Nuts
3          Wonka Bar - Milk Chocolate    Wicked Choccy's
4   Wonka Bar - Triple Dazzle Caramel    Wicked Choccy's
5                         Laffy Taffy        Sugar Shack
6                           SweeTARTS        Sugar Shack
7                               Nerds        Sugar Shack
8                             Fun Dip        Sugar Shack
9                Fizzy Lifting Drinks        Sugar Shack
10             Everlasting Gobstopper     Secret Factory
11                        Hair Toffee  The Other Factory
12                 Lickable Wallpaper     Secret Factory
13                          Wonka Gum     Secret Factory
14                          Kazookles  The Other Factory


In [4]:
factory_coordinates = pd.read_csv(
    "/content/factory_coordinates.csv"
)

print(factory_coordinates)

             Factory   Latitude   Longitude
0      Lot's O' Nuts  32.881893 -111.768036
1    Wicked Choccy's  32.076176  -81.088371
2        Sugar Shack  48.119140  -96.181150
3     Secret Factory  41.446333  -90.565487
4  The Other Factory  35.117500  -89.971107


In [5]:
model = joblib.load(
    "/content/best_model.pkl.pkl"
)

print("Model loaded successfully.")
print(type(model))

Model loaded successfully.
<class 'sklearn.pipeline.Pipeline'>


In [6]:
if hasattr(model, "feature_names_in_"):
    print("Model features:")
    print(model.feature_names_in_)
else:
    print(
        "The saved model does not expose feature_names_in_."
    )

Model features:
['Row ID' 'Order ID' 'Order Date' 'Ship Date' 'Ship Mode' 'Customer ID'
 'Country/Region' 'City' 'State/Province' 'Postal Code' 'Division'
 'Region' 'Product ID' 'Product Name' 'Sales' 'Units' 'Gross Profit'
 'Cost' 'Calculated Gross Profit' 'Profit Difference' 'Order Date Raw'
 'Ship Date Raw' 'Profit Margin' 'Sales per Unit' 'Profit per Unit'
 'Order Year' 'Order Month' 'Order Quarter' 'Order DayOfWeek']


In [7]:
print(model)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['Row ID', 'Customer ID',
                                                   'Sales', 'Units',
                                                   'Gross Profit', 'Cost',
                                                   'Calculated Gross Profit',
                                                   'Profit Difference',
                                                   'Profit Margin',
                                                   'Sales per Unit',
                                                   'Profit per Unit',
                                                   'Order Year', 'Order Month',
                                                   'Order Quarter',
                                                   'Order DayOfWeek']),
                                                 ('cat',
                                      

In [8]:
factory_map = dict(
    zip(
        product_factory["Product Name"],
        product_factory["Factory"]
    )
)

if "Factory" not in df.columns:
    df["Factory"] = df["Product Name"].map(factory_map)
else:
    df["Factory"] = df["Factory"].fillna(
        df["Product Name"].map(factory_map)
    )

print(
    df[
        ["Product Name", "Factory"]
    ].drop_duplicates()
)

                           Product Name            Factory
0            Wonka Bar - Milk Chocolate    Wicked Choccy's
1     Wonka Bar - Triple Dazzle Caramel    Wicked Choccy's
2     Wonka Bar - Nutty Crunch Surprise      Lot's O' Nuts
3        Wonka Bar -Scrumdiddlyumptious      Lot's O' Nuts
13            Wonka Bar - Fudge Mallows      Lot's O' Nuts
19                            Wonka Gum     Secret Factory
26                            Kazookles  The Other Factory
136                  Lickable Wallpaper     Secret Factory
191                Fizzy Lifting Drinks        Sugar Shack
226                         Laffy Taffy        Sugar Shack
973                           SweeTARTS        Sugar Shack
1490                              Nerds        Sugar Shack
4435                        Hair Toffee  The Other Factory
4729             Everlasting Gobstopper     Secret Factory
5007                            Fun Dip        Sugar Shack


In [9]:
factories = sorted(
    factory_coordinates["Factory"]
    .dropna()
    .unique()
)

print("Available factories:")
for factory in factories:
    print("-", factory)

Available factories:
- Lot's O' Nuts
- Secret Factory
- Sugar Shack
- The Other Factory
- Wicked Choccy's


In [10]:
baseline = (
    df.groupby(
        [
            "Product Name",
            "Region",
            "Ship Mode",
            "Factory"
        ],
        as_index=False
    )
    .agg(
        Orders=("Order ID", "nunique"),
        Units=("Units", "sum"),
        Sales=("Sales", "sum"),
        Gross_Profit=("Gross Profit", "sum"),
        Cost=("Cost", "sum"),
        Current_Lead_Time=("Lead Time Days", "mean")
    )
)

print(baseline.head())

             Product Name    Region       Ship Mode         Factory  Orders  \
0  Everlasting Gobstopper      Gulf  Standard Class  Secret Factory       1   
1  Everlasting Gobstopper  Interior  Standard Class  Secret Factory       1   
2  Everlasting Gobstopper   Pacific     First Class  Secret Factory       1   
3    Fizzy Lifting Drinks  Atlantic     First Class     Sugar Shack       1   
4    Fizzy Lifting Drinks  Atlantic  Standard Class     Sugar Shack       1   

   Units  Sales  Gross_Profit  Cost  Current_Lead_Time  
0      4  40.00         32.00   8.0             1273.0  
1      3  30.00         24.00   6.0             1641.0  
2      6  60.00         48.00  12.0             1270.0  
3      4  15.00          9.00   6.0             1637.0  
4      5  18.75         11.25   7.5             1274.0  


In [11]:
scenarios = []

for _, row in baseline.iterrows():

    current_factory = row["Factory"]

    for alternative_factory in factories:

        scenario = row.to_dict()

        scenario["Scenario_Factory"] = alternative_factory

        scenario["Is_Current"] = (
            alternative_factory == current_factory
        )

        scenarios.append(scenario)

scenario_df = pd.DataFrame(scenarios)

print("Scenario rows:", len(scenario_df))
print(scenario_df.head())

Scenario rows: 770
             Product Name Region       Ship Mode         Factory  Orders  \
0  Everlasting Gobstopper   Gulf  Standard Class  Secret Factory       1   
1  Everlasting Gobstopper   Gulf  Standard Class  Secret Factory       1   
2  Everlasting Gobstopper   Gulf  Standard Class  Secret Factory       1   
3  Everlasting Gobstopper   Gulf  Standard Class  Secret Factory       1   
4  Everlasting Gobstopper   Gulf  Standard Class  Secret Factory       1   

   Units  Sales  Gross_Profit  Cost  Current_Lead_Time   Scenario_Factory  \
0      4   40.0          32.0   8.0             1273.0      Lot's O' Nuts   
1      4   40.0          32.0   8.0             1273.0     Secret Factory   
2      4   40.0          32.0   8.0             1273.0        Sugar Shack   
3      4   40.0          32.0   8.0             1273.0  The Other Factory   
4      4   40.0          32.0   8.0             1273.0    Wicked Choccy's   

   Is_Current  
0       False  
1        True  
2       False

In [12]:
model_input = scenario_df[
    [
        "Product Name",
        "Scenario_Factory",
        "Region",
        "Ship Mode"
    ]
].copy()

In [13]:
print(model)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['Row ID', 'Customer ID',
                                                   'Sales', 'Units',
                                                   'Gross Profit', 'Cost',
                                                   'Calculated Gross Profit',
                                                   'Profit Difference',
                                                   'Profit Margin',
                                                   'Sales per Unit',
                                                   'Profit per Unit',
                                                   'Order Year', 'Order Month',
                                                   'Order Quarter',
                                                   'Order DayOfWeek']),
                                                 ('cat',
                                      

In [14]:
if hasattr(model, "feature_names_in_"):
    print(model.feature_names_in_)

['Row ID' 'Order ID' 'Order Date' 'Ship Date' 'Ship Mode' 'Customer ID'
 'Country/Region' 'City' 'State/Province' 'Postal Code' 'Division'
 'Region' 'Product ID' 'Product Name' 'Sales' 'Units' 'Gross Profit'
 'Cost' 'Calculated Gross Profit' 'Profit Difference' 'Order Date Raw'
 'Ship Date Raw' 'Profit Margin' 'Sales per Unit' 'Profit per Unit'
 'Order Year' 'Order Month' 'Order Quarter' 'Order DayOfWeek']


In [18]:
required_features = list(model.feature_names_in_)
model_input_for_prediction = pd.DataFrame(index=scenario_df.index) # Initialize empty DataFrame

# Define numerical and categorical columns based on the model's ColumnTransformer definition
numerical_cols = [
    'Row ID', 'Customer ID', 'Sales', 'Units', 'Gross Profit', 'Cost',
    'Calculated Gross Profit', 'Profit Difference', 'Profit Margin',
    'Sales per Unit', 'Profit per Unit', 'Order Year', 'Order Month',
    'Order Quarter', 'Order DayOfWeek'
]
categorical_cols = [
    'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Country/Region', 'City',
    'State/Province', 'Postal Code', 'Division', 'Region', 'Product ID',
    'Product Name', 'Order Date Raw', 'Ship Date Raw'
]

# Populate model_input_for_prediction ensuring all required_features are present and non-NaN
for col in required_features:
    if col == 'Gross Profit':
        # Handle the case where scenario_df has 'Gross_Profit' but model expects 'Gross Profit'
        if 'Gross_Profit' in scenario_df.columns:
            model_input_for_prediction[col] = scenario_df['Gross_Profit']
        else:
            model_input_for_prediction[col] = 0.0 # Default for numerical if not found
    elif col in scenario_df.columns:
        # If the column exists in scenario_df, copy its values
        model_input_for_prediction[col] = scenario_df[col]
    elif col in numerical_cols:
        # If a required numerical feature is not in scenario_df, fill with 0
        model_input_for_prediction[col] = 0.0
    elif col in categorical_cols:
        # If a required categorical feature is not in scenario_df, fill with an empty string
        model_input_for_prediction[col] = ''
    else:
        # Fallback for any unexpected required features (should not be hit if lists are complete)
        model_input_for_prediction[col] = '' # Default to empty string for safety

# Ensure the order of columns matches the model's expected order
model_input_for_prediction = model_input_for_prediction[required_features]

scenario_df["Predicted_Lead_Time"] = model.predict(model_input_for_prediction)

In [24]:
scenario_df["Profit_Margin"] = np.where(
    scenario_df["Sales"] != 0,
    scenario_df["Gross_Profit"]
    / scenario_df["Sales"],
    0
)

scenario_df["Lead_Time_Reduction"] = (
    scenario_df["Current_Lead_Time"]
    - scenario_df["Predicted_Lead_Time"]
)

scenario_df["Lead_Time_Reduction_%"] = np.where(
    scenario_df["Current_Lead_Time"] != 0,
    (
        scenario_df["Lead_Time_Reduction"]
        / scenario_df["Current_Lead_Time"]
    ) * 100,
    0
)

In [25]:
import os

output_dir = "../05_Results"
os.makedirs(output_dir, exist_ok=True)

scenario_df.to_csv(
    os.path.join(output_dir, "scenario_results.csv"),
    index=False
)

print("Scenario results saved.")

Scenario results saved.


In [26]:
from google.colab import files

output_path = os.path.join(output_dir, "scenario_results.csv")
files.download(output_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>